# 02 - Keşifsel veri analizi (EDA)

Temiz `cars` tablosunu (771 ilan) grafiklerle inceliyoruz. Fiyatlar USD, kilometre km cinsindendir.

**Hazırlık.** Proje köküne geçilir, matplotlib dosyaya çizim için `Agg` arka ucuna alınır.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
import matplotlib
matplotlib.use('Agg')
print(os.getcwd())

C:\Users\KAAN\Desktop\car-price-prediction\repository


**Veriyi yükle.** Veritabanından okur; şekil, eksik oranları ve fiyat özetini yazdırır.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.db.queries import load_cars

df = load_cars()
print(df.shape)
print(df.isna().mean().round(2).to_string())
print(df['price'].describe().round(0).to_string())
print('çarpıklık:', round(df['price'].skew(), 2))

(771, 13)
id              0.00
year            0.10
mileage         0.02
cylinders       0.32
brand           0.05
model           0.05
fuel_type       0.00
transmission    0.01
condition       0.36
drive           0.35
body_type       0.11
title_status    0.01
price           0.00
count       771.0
mean      15287.0
std       17027.0
min        1000.0
25%        5000.0
50%        9500.0
75%       18998.0
max      149980.0
çarpıklık: 3.17


**Grafik 1: fiyat dağılımı.** Ham fiyat ve log1p fiyat yan yana. Log dönüşümü sağa çarpık dağılımı simetrikleştirir; modeli neden log ölçeğinde eğittiğimizin gerekçesi budur.

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df['price'], bins=40, color='#3b6ea5')
axes[0].set_title('Fiyat dağılımı (ham)')
axes[0].set_xlabel('Fiyat (USD)')
axes[0].set_ylabel('İlan sayısı')
axes[1].hist(np.log1p(df['price']), bins=40, color='#c9772b')
axes[1].set_title('Fiyat dağılımı (log1p)')
axes[1].set_xlabel('log(1 + fiyat)')
axes[1].set_ylabel('İlan sayısı')
fig.savefig('docs/img/fiyat_dagilimi.png', dpi=150, bbox_inches='tight')
print(df['price'].quantile([.25, .5, .75, .95]).to_string())

0.25     5000.0
0.50     9500.0
0.75    18997.5
0.95    44652.5


Ham fiyat sağa çarpık: medyan 9.500 USD iken en pahalı ilan 150.000 USD'ye yakın; log ölçeğinde dağılım çok daha dengeli.

**Grafik 2: yıl ve fiyat.** Her nokta bir ilan; y ekseni log ölçeklidir ki ucuz ve pahalı arabalar aynı grafikte okunabilsin. Spearman korelasyonu da yazdırılır.

In [4]:
sub = df.dropna(subset=['year'])
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(sub['year'], sub['price'], s=12, alpha=0.5, color='#3b6ea5')
ax.set_yscale('log')
ax.set_title('Model yılı ve fiyat')
ax.set_xlabel('Model yılı')
ax.set_ylabel('Fiyat (USD, log ölçek)')
fig.savefig('docs/img/yil_fiyat.png', dpi=150, bbox_inches='tight')
print(sub['year'].corr(sub['price'], method='spearman'))

0.5567642215436024


Yeni model yıllı araçlar belirgin biçimde daha pahalı (Spearman yaklaşık 0,56); aynı yılda fiyat yine de geniş dağılıyor, yani yıl tek başına yetmiyor.

**Grafik 3: kilometre ve fiyat.** Aynı yöntem; kilometrenin 0'a çok yakın değerleri (veri hatası olasılığı) grafikte görünür.

In [5]:
sub = df.dropna(subset=['mileage'])
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(sub['mileage'], sub['price'], s=12, alpha=0.5, color='#c9772b')
ax.set_yscale('log')
ax.set_title('Kilometre ve fiyat')
ax.set_xlabel('Kilometre (km)')
ax.set_ylabel('Fiyat (USD, log ölçek)')
fig.savefig('docs/img/km_fiyat.png', dpi=150, bbox_inches='tight')
print(sub['mileage'].corr(sub['price'], method='spearman'))

-0.5298795042646867


Kilometre arttıkça fiyat düşüyor (Spearman yaklaşık -0,53); ilişki yıl kadar güçlü ama yıl ile kilometre birbirine bağlı olduğundan bilgilerin bir kısmı ortak.

**Grafik 4: markaya göre medyan fiyat.** En çok ilanı olan 15 marka seçilir; ortalama yerine medyan kullanılır çünkü birkaç aşırı pahalı ilan ortalamayı çarpıtır.

In [6]:
top = df['brand'].value_counts().head(15).index
med = df[df['brand'].isin(top)].groupby('brand')['price'].median().sort_values()
fig, ax = plt.subplots(figsize=(7, 5))
med.plot.barh(ax=ax, color='#3b6ea5')
ax.set_title('En sık 15 markanın medyan fiyatı')
ax.set_xlabel('Medyan fiyat (USD)')
ax.set_ylabel('Marka')
fig.savefig('docs/img/marka_medyan_fiyat.png', dpi=150, bbox_inches='tight')
print(med.sort_values(ascending=False).to_string())

brand
Tesla            27950.0
Chevrolet        14950.0
BMW              14247.5
Dodge            12000.0
Mercedes-Benz    11000.0
Jeep              9995.0
Lexus             9500.0
Ford              8950.0
Honda             8800.0
Toyota            8650.0
Hyundai           8500.0
GMC               7450.0
Kia               6999.0
Subaru            5500.0
Nissan            5050.0

Tesla (yaklaşık 27.950 USD) açık ara önde; Chevrolet ve BMW 14.000 USD üzerinde, Nissan ise yaklaşık 5.050 USD ile en ucuz grupta. Marka fiyat için güçlü ama bazı markalarda ilan sayısı az olduğundan medyanlar oynak.